# 📅 Day 01: Advanced Trie — Beyond Basic Prefix Search

---

## Trie Recap & What Makes It "Advanced"

Basic Trie was covered in Chapter 04 (Trees) — insert words, search prefixes. But Tries have superpowers way beyond simple autocomplete:

1. **Replace shortest root** — find shortest prefix that's a dictionary word
2. **Wildcard search** — match `.` as any character
3. **Binary Trie** — store numbers bit-by-bit to solve XOR problems

The key idea: instead of comparing whole strings, you **walk one character/bit at a time** through a tree. Each level = one character/bit of input.

```
╔═══════════════════════════════════════════════════════════════════════╗
║  TRIE = Trade space for speed on prefix operations                 ║
║                                                                     ║
║  String Trie: 26 children per node (a-z)                          ║
║  Binary Trie: 2 children per node (0-bit, 1-bit)                  ║
║                                                                     ║
║  Both give O(L) lookup where L = length of key                     ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# Trie building block — used in all problems below
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, word):
        node = self.root
        for c in word:
            if c not in node.children:
                node.children[c] = TrieNode()
            node = node.children[c]
        node.is_end = True

---

## 🎯 Problem 1: Replace Words (LC #648)

Given a dictionary of roots and a sentence, replace every word that has a root with the **shortest** root.

### Why Trie?

Checking every root against every word would be O(words × roots × length). With a Trie: build once from roots, then for each word walk the Trie until `is_end=True`. **First match = shortest root** since we go character by character.

```
  Trie from ["cat", "bat", "rat"]:
  
  root → c → a → t*
       → b → a → t*
       → r → a → t*
  
  Word "cattle": walk c→a→t → is_end=True → return "cat"!
  Word "the":    walk t → no 'h' child → return "the" unchanged
```

In [ ]:
def replaceWords(dictionary, sentence):
    """
    Build Trie from roots, find shortest prefix for each word.
    Time: O(sum of root lengths + sum of word lengths)
    """
    trie = Trie()
    for root in dictionary:
        trie.insert(root)
    
    def find_shortest_root(word):
        node = trie.root
        for i, c in enumerate(word):
            if c not in node.children:
                return word        # No root found
            node = node.children[c]
            if node.is_end:
                return word[:i+1]  # Found shortest root!
        return word
    
    return ' '.join(find_shortest_root(w) for w in sentence.split())

print(replaceWords(["cat","bat","rat"], "the cattle was rattled by the battery"))
# "the cat was rat by the bat"

---

## 🎯 Problem 2: Maximum XOR of Two Numbers (LC #421)

Find maximum `nums[i] XOR nums[j]` in an array.

### The Binary Trie Idea

XOR gives 1 when bits differ. To **maximize** XOR, at each bit position you want the **opposite** bit. This is exactly what a Trie enables — insert numbers bit-by-bit (MSB first), then for each number, greedily traverse the opposite bit at every level.

```
  nums = [3, 10, 5, 25, 2, 8] — in binary (5 bits):
    3  = 00011    25 = 11001
    5  = 00101
  
  For num=5 (00101), greedily pick opposite:
    bit 4: have 0, want 1 → 25 starts with 1! Take it.
    bit 3: have 0, want 1 → 25 has 1! Take it.
    bit 2: have 1, want 0 → 25 has 0! Take it.
    bit 1: have 0, want 1 → 25 has 0. No 1 path. Stuck.
    bit 0: have 1, want 0 → 25 has 1. No 0 path. Stuck.
    
  5 XOR 25 = 00101 XOR 11001 = 11100 = 28 → Answer!
```

In [ ]:
def findMaximumXOR(nums):
    """
    Binary Trie: for each number, greedily pick opposite bits.
    Time: O(32n) = O(n), Space: O(32n) = O(n)
    """
    root = {}
    
    # Insert all numbers into binary trie (MSB first)
    for num in nums:
        node = root
        for i in range(31, -1, -1):
            bit = (num >> i) & 1
            if bit not in node:
                node[bit] = {}
            node = node[bit]
    
    # For each number, find max XOR partner
    max_xor = 0
    for num in nums:
        node = root
        curr_xor = 0
        for i in range(31, -1, -1):
            bit = (num >> i) & 1
            want = 1 - bit  # Opposite for max XOR
            if want in node:
                curr_xor |= (1 << i)
                node = node[want]
            else:
                node = node[bit]
        max_xor = max(max_xor, curr_xor)
    
    return max_xor

print(findMaximumXOR([3,10,5,25,2,8]))  # 28 (5 XOR 25)

---

## 📊 Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  ADVANCED TRIE PATTERNS                                            ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                     ║
║  "shortest prefix/root"   → Trie + early return on is_end         ║
║  "wildcard '.' match"     → Trie + DFS on all children for '.'    ║
║  "max XOR"                → Binary Trie, greedily pick opposite   ║
║  "word search in board"   → Trie of words + DFS on grid          ║
║                                                                     ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 648 | Replace Words | Medium | Trie prefix lookup, early return |
| 421 | Maximum XOR of Two Numbers | Medium | Binary trie, greedy opposite |
| 211 | Design Add/Search Words | Medium | Trie + DFS for wildcards |
| 212 | Word Search II | Hard | Trie of words + grid DFS |
| 1268 | Search Suggestions System | Medium | Trie + DFS for top 3 |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 String Trie for prefix ops, Binary Trie for XOR problems      │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Shortest prefix: walk Trie, early return on is_end=True       │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Max XOR: greedily pick opposite bit at each level             │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Word Search II: build Trie from words, DFS the grid once      │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Segment Tree